In [1]:
# ① ライブラリ読み込み
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb

In [2]:

# =========================
# 実験名
# =========================
# 実験名セル
# 毎回ここだけ変える
EXPERIMENT_NAME = "exp001_base"

TARGET = "Irrigation_Need"
ID_COL = "id"

print("実験名:", EXPERIMENT_NAME)

実験名: exp001_base


In [3]:
# ② データ読み込み
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/test.csv")

print(train.shape)
print(test.shape)
display(train.head())

(630000, 21)
(270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [4]:
# ③ コピー作成
df_train = train.copy()
df_test = test.copy()

In [5]:
# ④ 目的変数・ID確認
target = "Irrigation_Need"
id_col = "id"

print(df_train.columns)
print(df_test.columns)
print(df_train[target].value_counts())

Index(['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm',
       'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage',
       'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare',
       'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need'],
      dtype='object')
Index(['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon',
       'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm',
       'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage',
       'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare',
       'Mulching_Used', 'Previous_Irrigation_mm', 'Region'],
      dtype='object')
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64


In [6]:
# ① 数値列ヒストグラム（一気に確認）
# ========================================
# 可視化①：ヒストグラム（数値全体）
# ========================================

# 何が分かる？
# 偏り（歪み）
# 外れ値
# 分布の形（正規っぽいか）

# import matplotlib.pyplot as plt

# num_cols = df_train.select_dtypes(include=["int64", "float64"]).columns

# df_train[num_cols].hist(figsize=(15, 10), bins=30)
# plt.suptitle("Numerical Features Distribution", fontsize=16)
# plt.show()

In [7]:
# ② ターゲット別ヒストグラム（超重要）
# ========================================
# 可視化②：ターゲット別分布
# ========================================
# ここが重要
# 分布がズレてる特徴量 = 強い特徴量候補

# for col in num_cols[:5]:  # 最初は5個だけでOK
#     plt.figure()
    
#     for label in sorted(df_train[target].unique()):
#         subset = df_train[df_train[target] == label]
#         plt.hist(subset[col], bins=30, alpha=0.5, label=f"class {label}")
    
#     plt.title(f"{col} by target")
#     plt.legend()
#     plt.show()

In [8]:
# ③ 相関ヒートマップ（全体把握）
# ========================================
# 可視化③：相関ヒートマップ
# ========================================
#  見るポイント
# 強い相関（0.8以上）
# ダブり特徴量（削除候補）

# import seaborn as sns

# plt.figure(figsize=(12, 10))
# corr = df_train[num_cols].corr()

# sns.heatmap(corr, cmap="coolwarm")
# plt.title("Correlation Heatmap")
# plt.show()

In [9]:
# ④ 1変数だけ深掘り（これ最強）
# ========================================
# 可視化④：単体チェック（重要）
# ========================================

# col = "Soil_Moisture"  # ←ここ変えるだけ

# plt.figure()
# for label in sorted(df_train[target].unique()):
#     subset = df_train[df_train[target] == label]
#     plt.hist(subset[col], bins=30, alpha=0.5, label=f"class {label}")

# plt.title(f"{col} distribution by target")
# plt.legend()
# plt.show()

In [10]:

#  使い方（超シンプル）

# ① ヒストグラム見る
# ② 「分布ズレてる列」探す
# ③ そこから特徴量作る

# あなた向けの見方（超重要）

# 例えば

# class0 → 低い
# class1 → 中間
# class2 → 高い

#  こうなってる列は
#  そのままでも強い or 掛け算でさらに強化できる

#  次の一手（ここから伸びる）

# このあと

#  「このグラフから何作る？」
#  一緒に“当たり特徴量”作れます

# スクショ出せば
#  ピンポイントで指示できます

In [11]:
# =========================
# 特徴量追加セル
# 毎回ここだけ変える
# =========================
features_new = []

# -------------------------
# 何も追加しないベース実験ならこのまま
# -------------------------

# -------------------------
# 追加例1: total_water
# -------------------------
# df_train["total_water"] = df_train["Rainfall_mm"] + df_train["Previous_Irrigation_mm"]
# df_test["total_water"] = df_test["Rainfall_mm"] + df_test["Previous_Irrigation_mm"]
# features_new.append("total_water")

# -------------------------
# 追加例2: water_balance
# 使うときは上を消してこれにする
# -------------------------
# df_train["water_balance"] = df_train["Rainfall_mm"] - df_train["Previous_Irrigation_mm"]
# df_test["water_balance"] = df_test["Rainfall_mm"] - df_test["Previous_Irrigation_mm"]
# features_new.append("water_balance")

# -------------------------
# 追加例3: moisture_x_temp
# -------------------------
# df_train["moisture_x_temp"] = df_train["Soil_Moisture"] * df_train["Temperature_C"]
# df_test["moisture_x_temp"] = df_test["Soil_Moisture"] * df_test["Temperature_C"]
# features_new.append("moisture_x_temp")

print("追加特徴量:", features_new)
print("df_train:", df_train.shape)
print("df_test :", df_test.shape)

追加特徴量: []
df_train: (630000, 21)
df_test : (270000, 20)


In [12]:
# =========================
# X, y 作成
# =========================
X = df_train.drop(columns=[TARGET]).copy()
y = df_train[TARGET].copy()

print("X:", X.shape)
print("y:", y.shape)

X: (630000, 20)
y: (630000,)


In [13]:
# =========================
# 目的変数エンコード
# =========================
target_le = LabelEncoder()
y = target_le.fit_transform(y)

print("target classes:", target_le.classes_)

target classes: ['High' 'Low' 'Medium']


In [14]:
# =========================
# カテゴリ変数エンコード
# =========================
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
print("カテゴリ列:", cat_cols)

for col in cat_cols:
    le = LabelEncoder()
    full_col = pd.concat([X[col], df_test[col]], axis=0).astype(str)
    le.fit(full_col)

    X[col] = le.transform(X[col].astype(str))
    df_test[col] = le.transform(df_test[col].astype(str))

print("Label Encoding 完了")

カテゴリ列: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
Label Encoding 完了


In [15]:
# =========================
# 学習用データ作成
# =========================
FEATURES = X.columns.tolist()

X_train = X.copy()
y_train = y.copy()
X_test = df_test[FEATURES].copy()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

X_train: (630000, 20)
X_test : (270000, 20)


In [16]:
# =========================
# CV学習
# =========================
num_classes = len(np.unique(y_train))
params = {
    "objective": "multiclass",
    "num_class": 3,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_data_in_leaf": 40,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "n_estimators": 300,   # ← ここが最重要
    "random_state": 42,
}


n_splits = 5
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

metrics = []
test_preds = np.zeros((len(X_test), num_classes))
imp = pd.DataFrame()

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
    print(f"\n===== Fold {fold} =====")

    x_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
    x_va, y_va = X_train.iloc[val_idx], y_train[val_idx]

    model = lgb.LGBMClassifier(**params)

    model.fit(
        x_tr, y_tr,
        eval_set=[(x_va, y_va)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(100),
        ],
    )

    y_tr_pred = model.predict(x_tr)
    y_va_pred = model.predict(x_va)

    acc_tr = accuracy_score(y_tr, y_tr_pred)
    acc_va = accuracy_score(y_va, y_va_pred)

    print(f"train_acc: {acc_tr:.5f}")
    print(f"val_acc  : {acc_va:.5f}")

    metrics.append([fold, acc_tr, acc_va])

    test_preds += model.predict_proba(X_test) / n_splits

    fold_imp = pd.DataFrame({
        "feature": X_train.columns,
        "importance": model.feature_importances_,
        "fold": fold,
    })
    imp = pd.concat([imp, fold_imp], axis=0, ignore_index=True)

metrics = np.array(metrics)

print("\n=== CV Result ===")
print(f"train mean: {metrics[:,1].mean():.5f}")
print(f"valid mean: {metrics[:,2].mean():.5f}")


===== Fold 1 =====
[LightGBM] [Warning] min_data_in_leaf is set=40, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=40
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[LightGBM] [Warning] lambda_l1 is set=0.1, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=40, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=40
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 w

In [17]:
# =========================
# 提出CSV作成
# =========================
pred_class = np.argmax(test_preds, axis=1)
pred_label = target_le.inverse_transform(pred_class)

submission = pd.DataFrame({
    ID_COL: df_test[ID_COL],
    TARGET: pred_label
})

save_name = f"submission_{EXPERIMENT_NAME}.csv"
submission.to_csv(save_name, index=False)

print(f"{save_name} を保存しました")
display(submission.head())

submission_exp001_base.csv を保存しました


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [18]:
# =========================
# 保存確認
# =========================
print(os.listdir("."))

['__notebook__.ipynb', 'submission_exp001_base.csv']


In [19]:
# # # 1回目
# EXPERIMENT_NAME = "exp001_base"

# # 特徴量追加なし

# 2回目
# EXPERIMENT_NAME = "exp002_total_water"

# # total_water 追加

# # 3回目
# EXPERIMENT_NAME = "exp003_water_balance"

# water_balance 追加

# 4回目
# EXPERIMENT_NAME = "exp004_moisture_x_temp"

# moisture_x_temp 追加

# もう一度、超重要
# 毎回変えるのは2か所だけ
# 変える場所①
# EXPERIMENT_NAME = "exp002_total_water"
# 変える場所②

# 特徴量追加セル

# それ以外は触らなくて大丈夫です。